In [1]:
#imports and connect
import os
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "road_accidents")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    print("Connected:", conn.execute(text("SELECT version();")).scalar())

Connected: PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [ ]:
# Load the cleaned dataset
df = pd.read_csv("../data/processed/accidents_clean.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(2)

Shape: (20000, 33)
Columns: ['accident_id', 'city', 'state', 'latitude', 'longitude', 'date', 'time', 'hour', 'day_of_week', 'is_weekend', 'road_type', 'lanes', 'traffic_signal', 'weather', 'visibility', 'temperature', 'traffic_density', 'cause', 'accident_severity', 'vehicles_involved', 'casualties', 'is_peak_hour', 'festival', 'risk_score', 'datetime', 'year', 'month', 'month_name', 'day', 'quarter', 'week_of_year', 'is_fatal', 'is_festival']


,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,risk_score,datetime,year,month,month_name,day,quarter,week_of_year,is_fatal,is_festival
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,05:00:00,5,Sunday,True,...,0.85,2023-10-22 05:00:00,2023,10,Oct,22,4,42,True,False
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,04:00:00,4,Sunday,True,...,0.10,2023-05-21 04:00:00,2023,5,May,21,2,20,False,False


In [ ]:
# Create the table in PostgreSQL using the SQL schema file
schema_sql = Path("../sql/01_create_tables.sql").read_text()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Table 'accidents' created with indexes.")

Table 'accidents' created with indexes.


In [4]:
# Prepare a copy so we don't mutate the original df
load_df = df.copy()

# Convert booleans to Python bool (pandas may pass numpy bools)
bool_cols = ["is_weekend", "traffic_signal", "is_peak_hour", "is_fatal", "is_festival"]
for col in bool_cols:
    load_df[col] = load_df[col].astype(bool)

# Postgres-safe: convert date/time/datetime to strings for clean insert
load_df["date"] = pd.to_datetime(load_df["date"]).dt.date
load_df["datetime"] = pd.to_datetime(load_df["datetime"])

load_df.to_sql(
    "accidents",
    engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)
print(f"Inserted {len(load_df)} rows.")

Inserted 20000 rows.


In [ ]:
# Validate row count in DB matches CSV
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM accidents;")).scalar()
    print("Rows in DB:", count)

if count == len(df):
    print("Row count matches")
else:
    print(f"⚠ Mismatch: CSV has {len(df)}, DB has {count}")

Rows in DB: 20000
Row count matches


In [11]:
# Spot check
pd.read_sql("""
SELECT accident_id, city, state, accident_severity, festival, is_festival, risk_score
FROM accidents
LIMIT 5;
""", engine)

,accident_id,city,state,accident_severity,festival,is_festival,risk_score
0,0,Pune,Maharashtra,Fatal,None,False,0.85
1,1,Mumbai,Maharashtra,Major,None,False,0.10
2,2,Mumbai,Maharashtra,Minor,None,False,0.45
3,3,Chandigarh,Punjab,Minor,None,False,0.65
4,4,Chennai,Tamil Nadu,Minor,None,False,0.10


In [12]:
# Quick KPI sanity check
pd.read_sql("""
SELECT
    COUNT(*)                                   AS total_accidents,
    SUM(casualties)                            AS total_casualties,
    ROUND(AVG(risk_score)::numeric, 2)         AS avg_risk,
    SUM(CASE WHEN is_fatal THEN 1 ELSE 0 END)  AS fatal_accidents
FROM accidents;
""", engine)

,total_accidents,total_casualties,avg_risk,fatal_accidents
0,20000,34529,0.44,2987
